In [0]:
import os
import sys

import mlflow
from pyspark.sql import functions as F

sys.path.insert(0, os.path.abspath(".."))
from modelos import pronostico as P
from modelos import semanal as S

dbutils.widgets.text("semanas", "8")  # el Job lo manda como parámetro; a mano vale este 8
SEMANAS = int(dbutils.widgets.get("semanas"))
NOMBRE = "vigia.ml.pronostico_dengue"
TABLA = "vigia.gold.pronosticos"
mlflow.set_registry_uri("databricks-uc")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLA} (
  dpa_provin STRING COMMENT 'Código DPA de la provincia, de 01 a 24',
  provincia STRING COMMENT 'Nombre de la provincia',
  emision DATE COMMENT 'Inicio (domingo) de la SE emitida: el pronóstico usa lo que se sabía al publicar su boletín',
  anio_epi BIGINT COMMENT 'Año epidemiológico de la SE emitida',
  semana_epi BIGINT COMMENT 'Número de la SE emitida',
  fecha_emision DATE COMMENT 'Sábado en que se publica el boletín, 7 días después de que cierra la SE emitida',
  horizonte BIGINT COMMENT 'Semanas hacia adelante, de 1 a 4',
  objetivo DATE COMMENT 'Inicio de la SE pronosticada',
  anio_objetivo BIGINT COMMENT 'Año epidemiológico de la SE pronosticada',
  semana_objetivo BIGINT COMMENT 'Número de la SE pronosticada',
  pronostico DOUBLE COMMENT 'Casos de dengue esperados en la SE pronosticada',
  pronostico_bajo DOUBLE COMMENT 'Extremo inferior del intervalo del 80 por ciento, según los errores del backtest',
  pronostico_alto DOUBLE COMMENT 'Extremo superior del intervalo del 80 por ciento',
  canal_mediana DOUBLE COMMENT 'Mediana del canal endémico en la SE pronosticada: encima empieza la zona de alerta',
  canal_q3 DOUBLE COMMENT 'Tercer cuartil del canal endémico en la SE pronosticada: encima empieza la zona epidémica',
  casos_semana DOUBLE COMMENT 'Casos de la SE emitida, como se conocían al emitir',
  casos_media_4s DOUBLE COMMENT 'Promedio de casos de las 4 semanas hasta la SE emitida',
  camas_dia DOUBLE COMMENT 'Camas-día que ocuparían los casos pronosticados; es un orden de magnitud',
  modelo STRING COMMENT 'Modelo campeón que pronosticó: gbm_global o gbm_sin_clima',
  version STRING COMMENT 'Versión de vigia.ml.pronostico_dengue que pronosticó',
  calculado_en TIMESTAMP COMMENT 'Cuándo se calculó la fila'
)
COMMENT 'Pronóstico semanal de casos de dengue por provincia a 1-4 semanas, con su intervalo y sus camas-día. Una fila por provincia, emisión y horizonte; cada sábado el Job agrega la emisión nueva'
""")

In [0]:
cliente = mlflow.MlflowClient()
version = cliente.get_model_version_by_alias(NOMBRE, "champion")
nombre = mlflow.get_run(version.run_id).data.params["modelo"]   # gbm_global o gbm_sin_clima
modelo = mlflow.sklearn.load_model(f"models:/{NOMBRE}@champion")

f = P.preparar(spark.table("vigia.gold.features_provincia_semana").toPandas())
hechas = spark.table(TABLA).select("emision").distinct().toPandas()["emision"]
hoy = spark.sql("SELECT current_date()").first()[0]
emisiones = S.emisiones_pendientes(f, hechas, hoy, SEMANAS)
print(f"Campeón: {nombre}, versión {version.version}. Por pronosticar: "
      f"{', '.join(str(e.date()) for e in emisiones) or 'nada, todo al día'}")

In [0]:
if emisiones:
    canal = spark.table("vigia.gold.canal_endemico").toPandas()
    semanas = (spark.table("vigia.gold.dim_calendario_epi")
               .select("anio_epi", "semana_epi", "inicio_semana").distinct().toPandas())
    backtest = P.de_spark(spark.table("vigia.ml.backtest_pronosticos").where(F.col("modelo") == nombre).toPandas())
    egresos = spark.table("vigia.silver.egresos_dengue").toPandas()
    casos = spark.table("vigia.gold.dengue_semana").select("dpa_provin", "anio_epi", "casos").toPandas()
    tasas = S.tasas_hospitalarias(egresos, casos)
    nuevos = S.pronosticar(modelo, f, canal, semanas, emisiones, S.factores_intervalo(backtest, nombre), tasas,
                           nombre, version.version)
    (spark.createDataFrame(P.para_spark(nuevos)).withColumn("calculado_en", F.current_timestamp())
     .write.mode("append").saveAsTable(TABLA))
    print(f"Camas-día: hospitalizaciones por caso y días de estada de {tasas['anios'].iloc[0]}")
    display(f[["dpa_provin", "provincia"]].drop_duplicates().merge(tasas, on="dpa_provin").round(3))

ultima = spark.table(TABLA).agg(F.max("emision")).first()[0]
dbutils.jobs.taskValues.set(key="emision", value=str(ultima))   # la tarea de la alerta la lee
display(spark.table(TABLA).where(F.col("emision") == ultima)
        .select("provincia", "horizonte", "semana_objetivo", "pronostico", "pronostico_bajo", "pronostico_alto",
                "canal_q3", "camas_dia")
        .orderBy("provincia", "horizonte"))